In [1]:
# Importing the libraries
import pandas as pd
import numpy as np
from utils.fake_na_detection_and_cleaning import FakeNullDetector
from utils.sql_connector import SQLConnector
from utils.data_type_converter import DataTypeConverter
from utils.normalizer import Normalizer
from utils.mapping_categorical import MappingCategorical

In [2]:
# loading of 2025 datasets and object creations
db=SQLConnector('Stack_Overflow_Survey')
db.connect()
query='select * from Bronze.Survey_2025'
Survey_2025_df = db.read_query(query)

FakeNullDetector_obj = FakeNullDetector()
DataTypeConverter_obj = DataTypeConverter()
Normalizer_obj = Normalizer()
MappingCategorical_obj = MappingCategorical()

Successfully Connected to Stack_Overflow_Survey
Query executed successfully


c:\Users\Ayush\Git Repo\Stack-Over-Flow-Survey-Data-Engineering-Project\Data Warehouse\Silver Layer\utils\sql_connector.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, self.conn)


In [3]:
# Null detector and remover 
Survey_2025_df_cleaned=Survey_2025_df.copy()
FakeNullDetector_obj.detect_fake_nulls(Survey_2025_df_cleaned)
FakeNullDetector_obj.replace_fake_nulls(Survey_2025_df_cleaned)

{'AIAcc': {'NA': np.int64(15894)},
 'AIAgentChallengesNeutral': {'NA': np.int64(27896)},
 'AIAgentChallengesSomewhat agree': {'NA': np.int64(27244)},
 'AIAgentChallengesSomewhat disagree': {'NA': np.int64(37066)},
 'AIAgentChallengesStrongly agree': {'NA': np.int64(26083)},
 'AIAgentChallengesStrongly disagree': {'NA': np.int64(39877)},
 'AIAgentChange': {'NA': np.int64(17513)},
 'AIAgentExtWrite': {'NA': np.int64(48330), 'None': np.int64(2)},
 'AIAgentExternal': {'NA': np.int64(40859)},
 'AIAgentImpactNeutral': {'NA': np.int64(39516)},
 'AIAgentImpactSomewhat agree': {'NA': np.int64(39045)},
 'AIAgentImpactSomewhat disagree': {'NA': np.int64(43400)},
 'AIAgentImpactStrongly agree': {'NA': np.int64(42504)},
 'AIAgentImpactStrongly disagree': {'NA': np.int64(45412)},
 'AIAgentKnowWrite': {'NA': np.int64(48424), 'None': np.int64(1)},
 'AIAgentKnowledge': {'NA': np.int64(45790)},
 'AIAgentObsWrite': {'N/A': np.int64(3),
                     'NA': np.int64(48918),
                     'Non

In [4]:
Survey_2025_df_cleaned['Employment'].value_counts()

In [5]:
# Mormalization of categorical columns : Basic mapping of values to reduce the number of unique values in each column and make it more consistent for analysis and visualization.
un_normalized_cols_name = ['Employment', 'EdLevel', 'Age', 'OpSysPersonal use', 'OpSysProfessional use', 'OrgSize', 'SOVisitFreq', 'SOAccount', 'SOPartFreq', 'SOComm', 'MainBranch']
normalized_cols_name = ['Employment', 'Education_Level', 'Age', 'OperatingSystem_Personal', 'OperatingSystem_Professional', 'Organization_Size', 'StackOverflow_Visit_Frequency', 'StackOverflow_Account_exists', 'StackOverflow_Participation_Frequency', 'StackOverflow_Community_Experience', 'Current_Profession']
columns_map = [
    MappingCategorical_obj.get_map('employment_map'),
    MappingCategorical_obj.get_map('ed_level_map'),
    MappingCategorical_obj.get_map('age_map'),
    MappingCategorical_obj.get_map('operating_system_map'),
    MappingCategorical_obj.get_map('operating_system_map'),
    MappingCategorical_obj.get_map('org_mapping'),
    MappingCategorical_obj.get_map('visit_freq_map'),
    MappingCategorical_obj.get_map('so_account_map'),
    MappingCategorical_obj.get_map('part_freq_map'),
    MappingCategorical_obj.get_map('comm_map'),
    MappingCategorical_obj.get_map('main_branch_map')
]
Survey_2025_df_cleaned = Normalizer_obj.normalize_categorical_columns_manual_mapping(Survey_2025_df_cleaned, un_normalized_cols_name, normalized_cols_name, columns_map)

['Employed' 'Independent contractor, freelancer, or self-employed'
 'Student' 'Retired' 'Not employed' 'I prefer not to say' nan]
Employment
Employed                       33750
Freelance                       6708
Student                         4428
Unemployed                      2227
Not Available or Applicable      852
Retired                          708
I prefer not to say              518
Name: count, dtype: int64
['MasterΓÇÖs degree (M.A., M.S., M.Eng., MBA, etc.)'
 'Associate degree (A.A., A.S., etc.)'
 'BachelorΓÇÖs degree (B.A., B.S., B.Eng., etc.)'
 'Some college/university study without earning a degree'
 'Professional degree (JD, MD, Ph.D, Ed.D, etc.)'
 'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)'
 'Other (please specify):' 'Primary/elementary school' nan]
Education_Level
Undergraduate                  28022
Postgraduate                   15213
High School                     3631
Not Available or Applicable     1042
Other         

In [6]:
# Mormalization of categorical columns : changing NA to more meaningful values and make it more consistent for analysis and visualization.
nan_replacer_columns=[]
cleaned_nan_replacer_columns=[]
if nan_replacer_columns:
    Survey_2025_df_cleaned=Normalizer_obj.normalize_na_replacer_columns(Survey_2025_df_cleaned,nan_replacer_columns, cleaned_nan_replacer_columns)

In [7]:
# Mormalization of categorical columns : Multi-select columns where respondents could select multiple options, resulting in semicolon-separated values.
multi_select_cols = []
multi_select_normalized = []
multi_select_maps = [

]
if multi_select_cols:
    Normalizer_obj.normalize_categorical_columns_non_exploding(Survey_2025_df_cleaned, multi_select_cols, multi_select_normalized, multi_select_maps)

In [8]:
# Mormalization of categorical columns : we will create a mapping to group similar roles and responses together, reducing the number of unique values while preserving the overall meaning.
tech_stack_cols = ['LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith', 'CommPlatformHaveWorkedWith', 'CommPlatformWantToWorkWith', 'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith', 'SOTagsHaveWorkedWith', 'SOTagsWantToWorkWith']
manual_mapping_cols = ['DevType', 'LearnCode']
all_target_cols = manual_mapping_cols + tech_stack_cols
manual_maps = {
    'DevType': MappingCategorical_obj.get_map('dev_type_map'),
    'LearnCode': MappingCategorical_obj.get_map('learn_code_map')
}
bridge_results = {}
target_names = [col + '_Clean' for col in all_target_cols]
maps_to_use = [manual_maps.get(col) for col in all_target_cols]
bridge_results = Normalizer_obj.normalize_categorical_columns_exploding(
    Survey_2025_df_cleaned, 
    all_target_cols, 
    target_names, 
    maps_to_use
)

--- Distribution for DevType_Clean ---
DevType_Clean
Other/Unknown          13454
Full-stack             12351
Back-end                6453
Student                 3008
Front-end               1974
Desktop/Enterprise      1919
Other                   1825
Mobile                  1391
Embedded/IoT            1274
Researcher              1131
Engineering Manager     1068
Data Engineer            770
Product Manager          507
SysAdmin                 480
Game/Graphics            451
DevOps                   441
Data/BI Analyst          351
QA/Testing               343
Name: count, dtype: int64
------------------------------
--- Distribution for LearnCode_Clean ---
LearnCode_Clean
Other/Unknown     147708
Physical Media     10187
Bootcamp            1747
Other               1068
Name: count, dtype: int64
------------------------------
--- Distribution for LanguageHaveWorkedWith_Clean ---
LanguageHaveWorkedWith_Clean
JavaScript                 21005
HTML/CSS                   19698
SQL  

In [9]:
# Cleaning the year Columns
experience_cols = ['YearsCode']
Survey_2025_df_cleaned = Normalizer_obj.clean_years_columns(Survey_2025_df_cleaned, experience_cols)

In [10]:
# Cleaning the Currency column and Salary Columns
col = 'Currency'
Survey_2025_df_cleaned[col] = Survey_2025_df_cleaned[col].str.split().str[0]
nan_replacer_cols = ['Currency']
cleaned_nan_cols = ['Currency_Code']
Survey_2025_df_cleaned = Normalizer_obj.normalize_na_replacer_columns(
    Survey_2025_df_cleaned,
    nan_replacer_cols, 
    cleaned_nan_cols,
    replacer_value="Not Available"
)
numeric_target_cols = ['CompTotal', 'ConvertedCompYearly']
Survey_2025_df_cleaned = DataTypeConverter_obj.string_to_numeric(Survey_2025_df_cleaned, numeric_target_cols)
Normalizer_obj.fill_na_and_remove_outlier_percentile_method(Survey_2025_df_cleaned, 'ConvertedCompYearly', 0.01, 0.95)
Normalizer_obj.fill_na_and_remove_outlier_percentile_method(Survey_2025_df_cleaned, 'CompTotal', 0.01, 0.95)

In [11]:
# Dropping the unrequired columns
# 1. Original columns that now have "Clean" versions
raw_redundant_cols = [col for col in ['Employment', 'EdLevel', 'Age', 'OpSysPersonal use', 'OpSysProfessional use', 'OrgSize', 'SOVisitFreq', 'SOAccount', 'SOPartFreq', 'SOComm', 'MainBranch'] if col != 'MainBranch'] + [] + [] + ['Currency']
# 2. Raw multi-select strings (already exploded into bridge_results)
multi_select_strings = all_target_cols
total_drop_list = list(set(raw_redundant_cols + multi_select_strings))
Survey_2025_df_cleaned.drop(columns=total_drop_list, inplace=True, errors='ignore')
print(f"Final Column Count: {len(Survey_2025_df_cleaned.columns)}")
print(Survey_2025_df_cleaned.columns.tolist())

Final Column Count: 152
['ResponseId', 'MainBranch', 'EmploymentAddl', 'WorkExp', 'LearnCodeChoose', 'LearnCodeAI', 'AILearnHow', 'YearsCode', 'ICorPM', 'RemoteWork', 'PurchaseInfluence', 'TechEndorseIntro', 'TechEndorse_1', 'TechEndorse_2', 'TechEndorse_3', 'TechEndorse_4', 'TechEndorse_5', 'TechEndorse_6', 'TechEndorse_7', 'TechEndorse_8', 'TechEndorse_9', 'TechEndorse_13', 'TechEndorse_13_TEXT', 'TechOppose_1', 'TechOppose_2', 'TechOppose_3', 'TechOppose_5', 'TechOppose_7', 'TechOppose_9', 'TechOppose_11', 'TechOppose_13', 'TechOppose_16', 'TechOppose_15', 'TechOppose_15_TEXT', 'Industry', 'JobSatPoints_1', 'JobSatPoints_2', 'JobSatPoints_3', 'JobSatPoints_4', 'JobSatPoints_5', 'JobSatPoints_6', 'JobSatPoints_7', 'JobSatPoints_8', 'JobSatPoints_9', 'JobSatPoints_10', 'JobSatPoints_11', 'JobSatPoints_13', 'JobSatPoints_14', 'JobSatPoints_15', 'JobSatPoints_16', 'JobSatPoints_15_TEXT', 'AIThreat', 'NewRole', 'ToolCountWork', 'ToolCountPersonal', 'Country', 'CompTotal', 'LanguageChoice

In [12]:
# Data Type Conversion
numeric_cols = ['YearsCode', 'CompTotal', 'ConvertedCompYearly', 'SurveyYear']
categorical_cols = ['MainBranch', 'Country', 'Education_Level', 'OperatingSystem_Personal', 'OperatingSystem_Professional', 'Organization_Size', 'StackOverflow_Visit_Frequency', 'StackOverflow_Account_exists', 'StackOverflow_Participation_Frequency', 'StackOverflow_Community_Experience', 'Current_Profession', 'Currency_Code']
DataTypeConverter_obj.string_to_category(Survey_2025_df_cleaned, categorical_cols)
DataTypeConverter_obj.string_to_numeric(Survey_2025_df_cleaned, numeric_cols)
Survey_2025_df_cleaned['SurveyYear'] = Survey_2025_df_cleaned['SurveyYear'].fillna(0).astype('datetime64[ns]')
print(Survey_2025_df_cleaned.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Columns: 152 entries, ResponseId to Currency_Code
dtypes: category(12), datetime64[ns](1), float64(2), int64(1), object(136)
memory usage: 53.2+ MB
None


In [13]:
for table_name, bridge_df in bridge_results.items():
    value_col = [col for col in bridge_df.columns if col != 'ResponseId'][0]
    bridge_results[table_name][value_col] = bridge_df[value_col].astype('category')
    bridge_results[table_name] = DataTypeConverter_obj.string_to_numeric(bridge_results[table_name], ['ResponseId'])
    print(f"Fixed types for {table_name}: {bridge_results[table_name][value_col].dtype}")

Fixed types for DevType_Clean: category
Fixed types for LearnCode_Clean: category
Fixed types for LanguageHaveWorkedWith_Clean: category
Fixed types for LanguageWantToWorkWith_Clean: category
Fixed types for DatabaseHaveWorkedWith_Clean: category
Fixed types for DatabaseWantToWorkWith_Clean: category
Fixed types for PlatformHaveWorkedWith_Clean: category
Fixed types for PlatformWantToWorkWith_Clean: category
Fixed types for WebframeHaveWorkedWith_Clean: category
Fixed types for WebframeWantToWorkWith_Clean: category
Fixed types for OfficeStackAsyncHaveWorkedWith_Clean: category
Fixed types for OfficeStackAsyncWantToWorkWith_Clean: category
Fixed types for AIModelsHaveWorkedWith_Clean: category
Fixed types for AIModelsWantToWorkWith_Clean: category
Fixed types for CommPlatformHaveWorkedWith_Clean: category
Fixed types for CommPlatformWantToWorkWith_Clean: category
Fixed types for DevEnvsHaveWorkedWith_Clean: category
Fixed types for DevEnvsWantToWorkWith_Clean: category
Fixed types for 

In [14]:
# Writing back to SQL
# Central Fact Table 2025
db.write_to_sql(df=Survey_2025_df_cleaned, schema='Silver', table_name='Survey_2025')
# Bridge Tables for Tech Stack and Manual Mapping Columns
for table_name, bridge_df in bridge_results.items():
    db.write_to_sql(df=bridge_df, schema='Silver', table_name=f"Bridge_{table_name}_2025")
db.close()

DataFrame written to Silver.Survey_2025 successfully.
DataFrame written to Silver.Bridge_DevType_Clean_2025 successfully.
DataFrame written to Silver.Bridge_LearnCode_Clean_2025 successfully.
DataFrame written to Silver.Bridge_LanguageHaveWorkedWith_Clean_2025 successfully.
DataFrame written to Silver.Bridge_LanguageWantToWorkWith_Clean_2025 successfully.
DataFrame written to Silver.Bridge_DatabaseHaveWorkedWith_Clean_2025 successfully.
DataFrame written to Silver.Bridge_DatabaseWantToWorkWith_Clean_2025 successfully.
DataFrame written to Silver.Bridge_PlatformHaveWorkedWith_Clean_2025 successfully.
DataFrame written to Silver.Bridge_PlatformWantToWorkWith_Clean_2025 successfully.
DataFrame written to Silver.Bridge_WebframeHaveWorkedWith_Clean_2025 successfully.
DataFrame written to Silver.Bridge_WebframeWantToWorkWith_Clean_2025 successfully.
DataFrame written to Silver.Bridge_OfficeStackAsyncHaveWorkedWith_Clean_2025 successfully.
DataFrame written to Silver.Bridge_OfficeStackAsyncWa

In [15]:
db.close()